In [2]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd

file_path = "archive/movies_metadata.csv"

df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "sopanmagar/moviesmetadata",
  file_path,
)

sample_n_rows = 20000
df = df.sample(n=min(sample_n_rows, len(df)), random_state=42).reset_index(drop=True)

100%|██████████| 32.8M/32.8M [00:00<00:00, 35.5MB/s]
/usr/local/lib/python3.12/dist-packages/kagglehub/pandas_datasets.py:92: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  result = read_function(


In [3]:
import pandas as pd
import random
import math
import numpy as np

m = df['vote_count'].quantile(0.90)
C = df['vote_average'].mean()

def weighted_rating(row, m=m, C=C):
    v = row['vote_count']
    R = row['vote_average']
    return round((v / (v + m)) * R + (m / (v + m)) * C, 2)

df['score'] = df.apply(weighted_rating, axis=1)
df = df.drop_duplicates(subset='title')
df = df.reset_index(drop=True)

cols_to_numeric = ['budget', 'popularity']

for col in cols_to_numeric:
    df[col] = pd.to_numeric(df[col], errors='coerce')


df['budget_log'] = np.log(df['budget'])
df['popularity_log'] = np.log(df['popularity'])

df.describe()

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4779: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)


,budget,popularity,revenue,runtime,vote_average,vote_count,score,budget_log,popularity_log
count,1.928800e+04,19287.000000,1.928700e+04,19172.000000,19287.000000,19287.000000,19287.000000,1.928800e+04,1.928700e+04
mean,4.101899e+06,2.892594,1.068426e+07,94.054767,5.603054,106.592627,5.715239,-inf,-inf
std,1.683033e+07,6.363521,5.966120e+07,37.865137,1.934286,464.178064,0.314216,NaN,NaN
min,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,0.000000,4.130000,-inf,-inf
25%,0.000000e+00,0.384263,0.000000e+00,85.000000,5.000000,3.000000,5.600000,NaN,-9.564294e-01
50%,0.000000e+00,1.120287,0.000000e+00,95.000000,6.000000,10.000000,5.620000,NaN,1.135849e-01
75%,0.000000e+00,3.637092,0.000000e+00,106.000000,6.800000,34.000000,5.710000,NaN,1.291184e+00
max,3.000000e+08,547.488298,1.513529e+09,1256.000000,10.000000,11187.000000,8.440000,1.951929e+01,6.305341e+00


In [4]:
df =df.drop(columns=['adult', 'belongs_to_collection','homepage','id', 'imdb_id','original_title','poster_path','production_countries', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'video', 'vote_count', 'vote_average', 'budget'])

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(df['overview'].fillna(''))

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
cosine_sim

indices = pd.Series(df.index, index=df['title']).drop_duplicates()

def get_recommendations(title, cosine_sim=cosine_sim, df=df, indices=indices):
    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:6]

    movie_indices = [i[0] for i in sim_scores]
    return df['title'].iloc[movie_indices]


In [6]:
get_recommendations('Toy Story')

,title
17602,Toy Story 3
3076,Small Fry
7391,Superstar: The Life and Times of Andy Warhol
11525,You're Only Young Once
17383,Malice


In [7]:
ratings_file_path = "archive/ratings_small.csv"

df_ratings = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "sopanmagar/moviesmetadata",
  ratings_file_path,
)

Using Colab cache for faster access to the 'moviesmetadata' dataset.


In [8]:
user_movie_matrix = df_ratings.pivot(
    index='userId',
    columns='movieId',
    values='rating'
).fillna(0)

In [9]:
def recommend(user_id, n=5):
    if user_id not in user_movie_matrix.index:
        popular = df_ratings.groupby('movieId')['rating'].mean().sort_values(ascending=False).head(n)
        return popular.index.tolist()

    user_sim = cosine_similarity(user_movie_matrix)
    user_sim_df = pd.DataFrame(user_sim, index=user_movie_matrix.index, columns=user_movie_matrix.index)

    similar_users = user_sim_df[user_id].sort_values(ascending=False)[1:21]

    watched = df_ratings[df_ratings['userId'] == user_id]['movieId'].values

    recommendations = {}
    for uid, sim in similar_users.items():
        user_ratings = df_ratings[df_ratings['userId'] == uid]
        new_movies = user_ratings[~user_ratings['movieId'].isin(watched)]

        for _, row in new_movies.iterrows():
            movie_id = row['movieId']
            rating = row['rating']
            if movie_id not in recommendations:
                recommendations[movie_id] = []
            recommendations[movie_id].append(rating * sim)

    avg_scores = {m: np.mean(s) for m, s in recommendations.items()}
    top_movies = pd.Series(avg_scores).sort_values(ascending=False).head(n)

    return top_movies.index.tolist()

In [10]:
print("Рекомендации для пользователя 1:")
print(recommend(1, 5))

print("\nРекомендации для нового пользователя:")
print(recommend(9999, 5))

Рекомендации для пользователя 1:
[36527.0, 69.0, 27478.0, 6883.0, 4085.0]

Рекомендации для нового пользователя:
[59273, 59549, 95377, 163949, 59447]
